# Build perfect-model X/Y from FOSI_BGC HR (prescribed-atmosphere t13 run)

Same convention as `3d_data_process.ipynb` sections 3a/3b/4, applied to the single
JRA55-forced HR hindcast instead of the CESM LE ensemble:

- **X, coarsened both ways** — the t13 native fields regridded down to the 1-degree
  regional grid via both `interp` (bilinear) and `avg` (conservative-average), saved as
  two separate files, matching `X_perfmodexp_SSP_interp.nc` / `X_perfmodexp_SSP_avg.nc`.
- **Y, high-res target** — `hi` regridded to the fine 0.1-degree regional grid, same
  role as `Y_perfmodexp_SSP.nc`: the "truth" half of a perfect-model pair, since X and Y
  both come from the same underlying HR simulation.

Differences from the existing pipeline, because of what this run actually is:
- **Single realization, not a 10-member ensemble** — `ensemble` dim is length 1 on both
  X and Y (kept on Y too, size 1, so `run_pipeline()`'s `(N, time, C, H, W)` shape
  convention still holds -- Y previously dropped this dim, which crashes
  `split_train_test`'s `N, T, Cy, Hy, Wy = Y.shape` unpacking).
- **`uatm`/`vatm` are grid-relative**, not true east/north — rotated via `ANGLET` before
  either regrid method, same order-of-operations reasoning as the rotation-check
  notebook. Output channels are `U_geo`/`V_geo` so it's unambiguous downstream.
- **No cm/s → m/s conversion** — `uatm`/`vatm` are already m/s.
- **Native grid pulled directly from the HR files**, not `pop_tools` — t13 isn't a
  registered grid there, but every monthly tseries file carries the full static grid.

Run this on Casper/Derecho with `/glade/campaign` mounted.

In [1]:
import glob
import warnings
import numpy as np
import xesmf as xe
import xarray as xr
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore", message="Latitude is outside of \\[-90, 90\\]")
warnings.filterwarnings("ignore", message="Input array is not C_CONTIGUOUS")

### 1. Collect files

In [2]:
RUN_DIR = Path(
    "/glade/campaign/cgd/oce/projects/FOSI_BGC/HR/g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001"
)
TSERIES_DIR = RUN_DIR / "ice" / "proc" / "tseries" / "month_1"

run_name = "FOSI_HR_JRA55"  # tag for provenance in the saved datasets
low_vars = ["hi", "aice", "uatm", "vatm"]  # X predictors
target_var = "hi"  # Y predictand

run_files = {}
for v in low_vars:
    files = sorted(TSERIES_DIR.glob(f"*.cice.h.{v}.*.nc"))
    run_files[v] = files
    print(f"{v}: {len(files)} files")

# uatm/vatm files should cover identical year ranges since they're written together
u_years = [f.name.split(".")[-2] for f in run_files["uatm"]]
v_years = [f.name.split(".")[-2] for f in run_files["vatm"]]
assert u_years == v_years, "uatm/vatm file year ranges don't line up -- check the glob."

# hi/aice/uatm/vatm should all cover the same record too, or channels will get
# silently truncated to the shortest one later -- check now rather than assume
lengths = {v: len(run_files[v]) for v in low_vars}
if len(set(lengths.values())) > 1:
    print("WARNING: mismatched file counts across variables:", lengths)
else:
    print("File counts match across all predictor variables:", lengths)

hi: 64 files
aice: 64 files
uatm: 64 files
vatm: 64 files
File counts match across all predictor variables: {'hi': 64, 'aice': 64, 'uatm': 64, 'vatm': 64}


### 2. Native grid + regridders (t13)

Same region as the existing pipeline (Kivalina domain) — keep `bbox_regrid`/`bbox` in
sync with `3d_data_process.ipynb` if that domain ever changes.

Three regridders, matching the three used in the original notebook for the high-res
CESM LE member:
- `regridder_hr_to_1deg_interp` — bilinear, for the X `interp` method
- `regridder_hr_to_1deg_cons` — conservative-average, for the X `avg` method (needs exact
  T-cell corners, built from U-points same as the original)
- `regridder_hr_to_0p1deg` — bilinear, for the Y high-res target

In [3]:
# ---------- Region select (keep in sync with 3d_data_process.ipynb) ----------

bbox_regrid = {"lon_min": -200, "lon_max": -130, "lat_min": 55, "lat_max": 85}
lon_min_regrid = bbox_regrid["lon_min"] % 360
lon_max_regrid = bbox_regrid["lon_max"] % 360

bbox = {"lon_min": -190, "lon_max": -140, "lat_min": 60, "lat_max": 80}
lon_min = bbox["lon_min"] % 360
lon_max = bbox["lon_max"] % 360

# ---------- Native grid, pulled straight from an HR file ----------

ds_grid = xr.open_dataset(run_files["hi"][0])
ds_grid = ds_grid.rename({"nj": "nlat", "ni": "nlon"})

for v in ["TLON", "TLAT", "ULON", "ULAT", "ANGLET"]:
    if v not in ds_grid:
        raise RuntimeError(f"Expected {v} in the HR history file, not found.")

tlon_full = ds_grid["TLON"].values % 360
tlat_full = ds_grid["TLAT"].values
ulon_full = ds_grid["ULON"].values % 360
ulat_full = ds_grid["ULAT"].values
angle_t = ds_grid["ANGLET"]  # T-grid angle - correct one for uatm/vatm (T-cell fields)
ny_full = tlon_full.shape[0]

mask_ice_hr = np.any(
    (tlat_full >= bbox_regrid["lat_min"])
    & (tlat_full <= bbox_regrid["lat_max"])
    & (tlon_full >= lon_min_regrid)
    & (tlon_full <= lon_max_regrid),
    axis=1,
)

grid_ice_hr = xr.Dataset({
    "lat": (["nlat", "nlon"], tlat_full[mask_ice_hr]),
    "lon": (["nlat", "nlon"], tlon_full[mask_ice_hr]),
})

print("Native t13 grid prepared.")

Native t13 grid prepared.


In [ ]:
# ---------- Conservative source grid (exact T-cell corners from U-points) ----------

rows = np.where(mask_ice_hr)[0]
r0, r1 = rows.min(), rows.max()

if not np.array_equal(rows, np.arange(r0, r1 + 1)):
    raise RuntimeError(
        "mask_ice_hr rows are not contiguous in nlat -- the corner "
        "construction below assumes a contiguous latitude band."
    )

if r0 < 1 or r1 + 1 > ny_full - 1:
    raise RuntimeError(
        "Masked region touches the native grid's j-edge/pole fold -- "
        "corner construction near the true boundary isn't handled here."
    )

ulon_i = np.pad(ulon_full, ((0, 0), (1, 0)), mode="wrap")
ulat_i = np.pad(ulat_full, ((0, 0), (1, 0)), mode="wrap")
lon_b_ice = ulon_i[r0 - 1:r1 + 1, :]
lat_b_ice = ulat_i[r0 - 1:r1 + 1, :]

grid_ice_hr_conserv = xr.Dataset({
    "lat": (["nlat", "nlon"], tlat_full[r0:r1 + 1, :]),
    "lon": (["nlat", "nlon"], tlon_full[r0:r1 + 1, :]),
    "lat_b": (["nlat_b", "nlon_b"], lat_b_ice),
    "lon_b": (["nlat_b", "nlon_b"], lon_b_ice),
})

print("Ice conservative source grid built (exact t13 U-point corners).")

# ---------- Destination grids ----------

dst_1deg = xr.Dataset({
    "lat": ("lat", np.arange(bbox_regrid["lat_min"], bbox_regrid["lat_max"] + 1, 1.0)),
    "lon": ("lon", np.arange(lon_min_regrid, lon_max_regrid + 1, 1.0)),
})

dst_lat_c, dst_lon_c = dst_1deg.lat.values, dst_1deg.lon.values
dst_lat_b = np.concatenate([[dst_lat_c[0] - 0.5], dst_lat_c + 0.5])
dst_lon_b = np.concatenate([[dst_lon_c[0] - 0.5], dst_lon_c + 0.5])

dst_1deg_b = xr.Dataset({
    "lat": ("lat", dst_lat_c),
    "lon": ("lon", dst_lon_c),
    "lat_b": ("lat_b", dst_lat_b),
    "lon_b": ("lon_b", dst_lon_b),
})

dst_0p1deg = xr.Dataset({
    "lat": ("lat", np.arange(bbox_regrid["lat_min"], bbox_regrid["lat_max"] + 0.1, 0.1)),
    "lon": ("lon", np.arange(lon_min_regrid, lon_max_regrid + 0.1, 0.1)),
})

# ---------- Regridders ----------

print("Building/locating regridders...")

WEIGHTED_GRIDS_DIR = "/glade/work/skygale/_projects/SeaIceDownscaling/weighted_grids"

regridder_hr_to_1deg_interp = xe.Regridder(
    grid_ice_hr, dst_1deg, method="bilinear", periodic=True,
    filename=f"{WEIGHTED_GRIDS_DIR}/hr_t13_to_1deg_interp.nc", reuse_weights=False,
)
print(" >> Built regridder_hr_to_1deg_interp.")

regridder_hr_to_1deg_cons = xe.Regridder(
    grid_ice_hr_conserv, dst_1deg_b, method="conservative", periodic=True,
    filename=f"{WEIGHTED_GRIDS_DIR}/hr_t13_to_1deg_cons.nc", reuse_weights=False,
)
print(" >> Built regridder_hr_to_1deg_cons.")

regridder_hr_to_0p1deg = xe.Regridder(
    grid_ice_hr, dst_0p1deg, method="bilinear", periodic=True,
    filename=f"{WEIGHTED_GRIDS_DIR}/hr_t13_to_0p1deg.nc", reuse_weights=False,
)
print(" >> Built regridder_hr_to_0p1deg.")

### 3. Processing functions

Both take a `regridder` argument so the same functions serve `interp`, `avg`, and the
high-res Y regrid — only the regridder object passed in changes.

In [6]:
def process_scalar(file, var, regridder):
    """hi or aice: mask to region, regrid, fill/cast."""
    ds = xr.open_dataset(file)
    da = ds[var].rename({"nj": "nlat", "ni": "nlon"})
    da = da.isel(nlat=mask_ice_hr)

    da_reg = regridder(da)
    da_reg = da_reg.sel(lat=slice(bbox["lat_min"], bbox["lat_max"]),
                         lon=slice(lon_min, lon_max))
    da_reg = da_reg.fillna(0).astype(np.float32)
    ds.close()
    return da_reg


def process_wind_pair(u_file, v_file, angle, regridder):
    """uatm/vatm: rotate grid-relative -> true E/N using ANGLET, then regrid."""
    ds_u = xr.open_dataset(u_file).rename({"nj": "nlat", "ni": "nlon"})
    ds_v = xr.open_dataset(v_file).rename({"nj": "nlat", "ni": "nlon"})

    u_geo = ds_u["uatm"] * np.cos(angle) - ds_v["vatm"] * np.sin(angle)
    v_geo = ds_u["uatm"] * np.sin(angle) + ds_v["vatm"] * np.cos(angle)

    u_geo = u_geo.isel(nlat=mask_ice_hr)
    v_geo = v_geo.isel(nlat=mask_ice_hr)

    u_reg = regridder(u_geo)
    v_reg = regridder(v_geo)

    u_reg = u_reg.sel(lat=slice(bbox["lat_min"], bbox["lat_max"]),
                       lon=slice(lon_min, lon_max)).fillna(0).astype(np.float32)
    v_reg = v_reg.sel(lat=slice(bbox["lat_min"], bbox["lat_max"]),
                       lon=slice(lon_min, lon_max)).fillna(0).astype(np.float32)

    ds_u.close()
    ds_v.close()
    return u_reg, v_reg

### 4. Build X (both regrid methods)

Same `hi`/`aice`/`U_geo`/`V_geo` channel stack, run once per method, saved as two
separate files -- matches `X_perfmodexp_SSP_interp.nc` / `X_perfmodexp_SSP_avg.nc`.

In [7]:
regridders = {
    "interp": regridder_hr_to_1deg_interp,
    "avg": regridder_hr_to_1deg_cons,
}
channel_order = ["hi", "aice", "U_geo", "V_geo"]

for method, regridder in regridders.items():
    print(f"\n=== Building X, method={method} ===")

    channels = {}
    for var in ["hi", "aice"]:
        print(f"Processing {var}...")
        parts = [process_scalar(f, var, regridder) for f in run_files[var]]
        channels[var] = xr.concat(parts, dim="time")

    print("Processing uatm/vatm (with ANGLET rotation)...")
    u_parts, v_parts = [], []
    for uf, vf in zip(run_files["uatm"], run_files["vatm"]):
        u_reg, v_reg = process_wind_pair(uf, vf, angle_t, regridder)
        u_parts.append(u_reg)
        v_parts.append(v_reg)
    channels["U_geo"] = xr.concat(u_parts, dim="time")
    channels["V_geo"] = xr.concat(v_parts, dim="time")

    min_t = min(channels[c].sizes["time"] for c in channel_order)
    stacked = [channels[c].isel(time=slice(0, min_t)) for c in channel_order]

    X_ds = xr.concat(stacked, dim="channel")
    X_ds.name = "X"
    X_ds = X_ds.assign_coords(channel=channel_order)
    X_ds = X_ds.expand_dims({"ensemble": [0]})  # single realization, kept for shape parity

    X_ds.attrs["description"] = (
        f"Prescribed-atmosphere (JRA55-forced) t13 hindcast predictors, regridded to "
        f"the 1-degree regional grid via {method}. Single realization, not an ensemble."
    )
    X_ds.attrs["source_run"] = run_name
    X_ds.attrs["regrid_method"] = method
    X_ds.attrs["notes"] = (
        "uatm/vatm rotated from grid-relative to true east/north via ANGLET before "
        "regridding; NaNs filled with zero."
    )
    X_ds.attrs["created_by"] = "Sky Gale"
    X_ds.attrs["date_created"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    X_ds.attrs["variables"] = (
        "hi: sea ice thickness (m); "
        "aice: sea ice concentration; "
        "U_geo: true-east wind velocity (m s-1), rotated from uatm; "
        "V_geo: true-north wind velocity (m s-1), rotated from vatm"
    )

    X_ds = X_ds.transpose("ensemble", "time", "channel", "lat", "lon")

    save_path = f"/glade/derecho/scratch/skygale/Downscaling_Data/X_{run_name}_{method}.nc"
    X_ds.to_netcdf(save_path)
    print("Saved to:", save_path)


=== Building X, method=interp ===
Processing hi...
Processing aice...
Processing uatm/vatm (with ANGLET rotation)...
Saved to: /glade/derecho/scratch/skygale/Downscaling_Data/X_FOSI_HR_JRA55_interp.nc

=== Building X, method=avg ===
Processing hi...
Processing aice...
Processing uatm/vatm (with ANGLET rotation)...
Saved to: /glade/derecho/scratch/skygale/Downscaling_Data/X_FOSI_HR_JRA55_avg.nc


### 5. Build Y (high-res target)

`hi` only, regridded to the fine 0.1-degree regional grid — the "truth" half of the
perfect-model pair, matching `Y_perfmodexp_SSP.nc`.

In [ ]:
print(f"=== Building Y, target_var={target_var} ===")

y_parts = [process_scalar(f, target_var, regridder_hr_to_0p1deg) for f in run_files[target_var]]
Y_da = xr.concat(y_parts, dim="time")
Y_da.name = "Y"

Y_ds = Y_da.expand_dims({"channel": [0]})
Y_ds = Y_ds.expand_dims({"ensemble": [0]})  # kept for shape parity with X (see run_pipeline)
Y_ds = Y_ds.transpose("ensemble", "time", "channel", "lat", "lon")

Y_ds.attrs["description"] = (
    "Prescribed-atmosphere (JRA55-forced) t13 hindcast, hi regridded to the fine "
    "0.1-degree regional grid -- perfect-model target paired with X_{run}_*.nc."
).format(run=run_name)
Y_ds.attrs["source_run"] = run_name
Y_ds.attrs["notes"] = "NaNs filled with zero."
Y_ds.attrs["created_by"] = "Sky Gale"
Y_ds.attrs["date_created"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
Y_ds.attrs["variables"] = "hi: sea ice thickness (m)"

save_path = f"/glade/derecho/scratch/skygale/Downscaling_Data/Y_{run_name}.nc"
Y_ds.to_netcdf(save_path)
print("Saved to:", save_path)

### 6. Quick checks

Shapes against the existing MESACLIP tensors (same `channel`/spatial extent expected;
`ensemble` differs: 1 vs 10), plus a tight-domain wind quiver check zoomed to the actual
regional bbox so individual arrows are visible rather than rendering as solid texture.

In [ ]:
X_interp = xr.open_dataset(f"/glade/derecho/scratch/skygale/Downscaling_Data/X_{run_name}_interp.nc").X
X_avg = xr.open_dataset(f"/glade/derecho/scratch/skygale/Downscaling_Data/X_{run_name}_avg.nc").X
Y_check = xr.open_dataset(f"/glade/derecho/scratch/skygale/Downscaling_Data/Y_{run_name}.nc").Y
print("X interp:", X_interp.shape, X_interp.channel.values)
print("X avg:   ", X_avg.shape, X_avg.channel.values)
print("Y:       ", Y_check.shape)

X_mesaclip = xr.open_dataset(
    "/glade/derecho/scratch/skygale/Downscaling_Data/X_perfmodexp_SSP_avg.nc"
).X
Y_mesaclip = xr.open_dataset(
    "/glade/derecho/scratch/skygale/Downscaling_Data/Y_perfmodexp_SSP.nc"
).Y
print("MESACLIP X avg:", X_mesaclip.shape, X_mesaclip.channel.values)
print("MESACLIP Y:    ", Y_mesaclip.shape)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

t = 0
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.set_extent([bbox["lon_min"], bbox["lon_max"], bbox["lat_min"], bbox["lat_max"]],
              crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.coastlines()

lon2d, lat2d = np.meshgrid(X_avg.lon.values, X_avg.lat.values)
u = X_avg.sel(channel="U_geo").isel(ensemble=0, time=t).values
v = X_avg.sel(channel="V_geo").isel(ensemble=0, time=t).values

ax.quiver(lon2d, lat2d, u, v, transform=ccrs.PlateCarree())
ax.set_title(f"Rotated + regridded wind (avg method), t={t}")
plt.show()